# Regression test suite for podaac-l2-subsetter

This notebook tests the [podaac-l2-subsetter](https://github.com/podaac/l2ss-py) service,
which provides Level 2 spatial and variable subsetting via Harmony.

### Features tested:

* Bounding box spatial subsetting
* Variable subsetting
* Temporal subsetting
* Shapefile subsetting

### Prerequisites

The dependencies for this notebook are listed in the [environment.yaml](./environment.yaml).
To test or install locally, create the papermill environment used in the automated regression testing suite:

`conda env create -f ./environment.yaml && conda activate papermill-podaac-l2-subsetter`

A `.netrc` file must also be located in the `test` directory of this repository.

# Setup

## Import required packages:

In [ ]:
from os.path import exists
from pathlib import Path
from tempfile import TemporaryDirectory

from earthdata_hashdiff import nc4_matches_reference_hash_file
from harmony import BBox, Client, Collection, Environment, Request


class SkipAllAttributes(set):
    """A set that contains everything — used to skip all metadata attributes."""

    def __contains__(self, item):
        return True

### Import shared utility functions:

In [ ]:
import sys

sys.path.append('../shared_utils')
from utilities import print_success, submit_and_download

## Set default parameters:

`papermill` requires default values for parameters used on the workflow. In this case, `harmony_host_url`.

In [ ]:
harmony_host_url = 'https://harmony.uat.earthdata.nasa.gov'

### Identify Harmony environment:

In [ ]:
host_environment = {
    'http://localhost:3000': Environment.LOCAL,
    'https://harmony.sit.earthdata.nasa.gov': Environment.SIT,
    'https://harmony.uat.earthdata.nasa.gov': Environment.UAT,
    'https://harmony.earthdata.nasa.gov': Environment.PROD,
}

harmony_environment = host_environment.get(harmony_host_url)

if harmony_environment is not None:
    harmony_client = Client(env=harmony_environment)

## Test data configuration

Define test collections and granules per environment.

In [ ]:
l2ss_test_data_uat = {
    'collection': Collection(id='C1234724470-POCLOUD'),
}

l2ss_test_data_prod = {
    'collection': Collection(id='C1940473819-POCLOUD'),
}

l2ss_test_data_by_environment = {
    Environment.LOCAL: l2ss_test_data_uat,
    Environment.SIT: l2ss_test_data_uat,
    Environment.UAT: l2ss_test_data_uat,
    Environment.PROD: l2ss_test_data_prod,
}

if harmony_environment in l2ss_test_data_by_environment:
    l2ss_test_data = l2ss_test_data_by_environment[harmony_environment]
else:
    l2ss_test_data = None

In [ ]:
request_info = {
    **l2ss_test_data,
    'spatial': BBox(-160, 30, -120, 60),
    'variables': [
        'l2p_flags',
        'sea_surface_temperature',
    ],
    'granule_name': [
        '20250717011501-JPL-L2P_GHRSST-SSTskin-MODIS_A-D-v02.0-fv01.0'
    ],
} if l2ss_test_data is not None else None

## Test Execution Helper

In [ ]:
def execute_l2ss_test(
    harmony_client_object,
    harmony_request,
    request_name,
    output_filename,
    reference_filepath,
):
    """Execute a podaac-l2-subsetter test with temporary directory handling and validation."""
    with TemporaryDirectory() as tmp_dir:
        output_path = tmp_dir / Path(output_filename)
        submit_and_download(harmony_client_object, harmony_request, output_path)
        assert exists(output_path), f'Unsuccessful {request_name}.'
        assert nc4_matches_reference_hash_file(
            output_path,
            reference_filepath,
            skipped_metadata_attributes=SkipAllAttributes(),
        ), f'{request_name}: Output and reference files do not match'

    print_success(request_name)

# Begin regression tests

## Test 1: Bounding box spatial subsetting

**Tests:** Spatial subsetting with a bounding box on a single granule.

In [ ]:
if request_info is not None:
    bbox_request = Request(
        collection=request_info['collection'],
        granule_name=request_info['granule_name'][0],
        spatial=request_info['spatial'],
        labels=['podaac-l2-subsetter-rtests', 'podaac-l2-subsetter-rtest1'],
    )

    execute_l2ss_test(
        harmony_client,
        bbox_request,
        request_name='podaac-l2-subsetter bounding box spatial subset',
        output_filename='bbox_subset.nc4',
        reference_filepath='reference_files/bbox_subset.json',
    )
else:
    print(
        f'podaac-l2-subsetter is not configured for environment: "{harmony_environment}" - skipping test.'
    )

## Test 2: Variable subsetting

**Tests:** Variable subsetting on a single granule.

In [ ]:
if request_info is not None:
    variable_request = Request(
        collection=request_info['collection'],
        granule_name=request_info['granule_name'][0],
        variables=request_info['variables'],
        labels=['podaac-l2-subsetter-rtests', 'podaac-l2-subsetter-rtest2'],
    )

    execute_l2ss_test(
        harmony_client,
        variable_request,
        request_name='podaac-l2-subsetter variable subset',
        output_filename='var_subset.nc4',
        reference_filepath='reference_files/var_subset.json',
    )
else:
    print(
        f'podaac-l2-subsetter is not configured for environment: "{harmony_environment}" - skipping test.'
    )

## Test 3: Combined spatial and variable subsetting

**Tests:** Both bounding box and variable subsetting on a single granule.

In [ ]:
if request_info is not None:
    combined_request = Request(
        collection=request_info['collection'],
        granule_name=request_info['granule_name'][0],
        spatial=request_info['spatial'],
        variables=request_info['variables'],
        labels=['podaac-l2-subsetter-rtests', 'podaac-l2-subsetter-rtest3'],
    )

    execute_l2ss_test(
        harmony_client,
        combined_request,
        request_name='podaac-l2-subsetter spatial + variable subset',
        output_filename='bbox_var_subset.nc4',
        reference_filepath='reference_files/bbox_var_subset.json',
    )
else:
    print(
        f'podaac-l2-subsetter is not configured for environment: "{harmony_environment}" - skipping test.'
    )

## Test 4: No subsetting (passthrough)

**Tests:** Single granule with no subsetting parameters — verifies passthrough behavior.

In [ ]:
if request_info is not None:
    passthrough_request = Request(
        collection=request_info['collection'],
        granule_name=request_info['granule_name'][0],
        labels=['podaac-l2-subsetter-rtests', 'podaac-l2-subsetter-rtest4'],
    )

    execute_l2ss_test(
        harmony_client,
        passthrough_request,
        request_name='podaac-l2-subsetter no subset passthrough',
        output_filename='passthrough.nc4',
        reference_filepath='reference_files/passthrough.json',
    )
else:
    print(
        f'podaac-l2-subsetter is not configured for environment: "{harmony_environment}" - skipping test.'
    )